## Montecarlo Claude

In [1]:
"""
Monte Carlo Suite per Trading Systems
======================================

Suite completa di metodi Monte Carlo per validazione trading systems:
1. Trade-Based Bootstrap (raccomandato per TS)
2. Block Bootstrap Returns (preserva autocorrelazione)
3. Regime-Switching Bootstrap
4. Simple Bootstrap Corrected (baseline migliorato)

Include analisi completa, visualizzazioni e interpretazione automatica.

Autore: Framework Analysis Project
Data: Febbraio 2026
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Union
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


# =============================================================================
# 1. TRADE-BASED BOOTSTRAP (RACCOMANDATO PER TS)
# =============================================================================

def monte_carlo_trade_bootstrap(
    trades_df: pd.DataFrame,
    benchmark_return: Optional[float] = None,  # NUOVO
    benchmark_drawdown: Optional[float] = None,  # NUOVO
    n_simulations: int = 10_000,
    init_cash: float = 100_000,
    pnl_column: str = 'PnL',
    random_seed: Optional[int] = None,
    show_progress: bool = True
) -> Dict:
    """
    Monte Carlo Bootstrap basato su TRADE invece di returns giornalieri.
    
    Resample i trade preservando distribuzione PnL ma randomizzando ordine.
    MIGLIORE per trading systems perché:
    - Preserva distribuzione asimmetrica PnL trade
    - Identifica dipendenza da pochi trade lucky
    - Realistico per systems con <200 trade/anno
    
    Parametri
    ----------
    trades_df : pd.DataFrame
        DataFrame trade con almeno colonna PnL
        (da VectorBT: portfolio.trades.records_readable)
    n_simulations : int, default=10_000
        Numero simulazioni
    init_cash : float, default=100_000
        Capitale iniziale
    pnl_column : str, default='PnL'
        Nome colonna PnL nel DataFrame
    random_seed : int, optional
        Seed per riproducibilità
    show_progress : bool, default=True
        Progress bar
        
    Returns
    -------
    dict
        Risultati con equity curves, metriche, streaks
    """
    
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Estrai PnL trade
    if pnl_column not in trades_df.columns:
        raise ValueError(f"Colonna '{pnl_column}' non trovata in trades_df")
    
    trades_pnl = trades_df[pnl_column].dropna().values
    n_trades = len(trades_pnl)
    
    if n_trades == 0:
        raise ValueError("Nessun trade con PnL valido")
    
    # Storage risultati
    equity_curves = []
    final_returns = []
    max_drawdowns = []
    sharpe_ratios = []
    win_streaks = []
    loss_streaks = []
    
    # NUOVO: Benchmark comparisons
    below_benchmark_return_count = 0
    above_benchmark_drawdown_count = 0

    # Simulazioni
    pbar = tqdm(total=n_simulations, desc="Trade Bootstrap", disable=not show_progress)
    
    for _ in range(n_simulations):
        # Resample trade con replacement
        sampled_trades = np.random.choice(trades_pnl, size=n_trades, replace=True)
        
        # Costruisci equity curve
        equity = init_cash + np.cumsum(sampled_trades)
        equity = np.insert(equity, 0, init_cash)  # aggiungi valore iniziale
        
        # Metriche
        final_value = equity[-1]
        final_return = (final_value / init_cash) - 1
        
        # Drawdown
        cummax = np.maximum.accumulate(equity)
        drawdown = (equity - cummax) / cummax
        max_dd = abs(drawdown.min())
        
        # Sharpe (sui trade PnL, non equity)
        if len(sampled_trades) > 1:
            sharpe = (np.mean(sampled_trades) / (np.std(sampled_trades) + 1e-8)) * np.sqrt(252 / max(1, n_trades / 365))
        else:
            sharpe = 0.0
        
        # Win/Loss streaks
        win_mask = sampled_trades > 0
        max_win_streak = _max_consecutive(win_mask)
        max_loss_streak = _max_consecutive(~win_mask)
        
        # Store
        equity_curves.append(equity)
        final_returns.append(final_return)
        max_drawdowns.append(max_dd)
        sharpe_ratios.append(sharpe)
        win_streaks.append(max_win_streak)
        loss_streaks.append(max_loss_streak)

        # NUOVO: Benchmark comparisons
        if benchmark_return is not None and final_return < benchmark_return:
            below_benchmark_return_count += 1
        if benchmark_drawdown is not None and max_dd > benchmark_drawdown:
            above_benchmark_drawdown_count += 1

        pbar.update(1)
    
    pbar.close()
    
    # Statistiche aggregate
    return {
        'method': 'Trade-Based Bootstrap',
        'n_simulations': n_simulations,
        'n_trades': n_trades,
        'equity_curves': equity_curves,
        'final_returns': np.array(final_returns),
        'max_drawdowns': np.array(max_drawdowns),
        'sharpe_ratios': np.array(sharpe_ratios),
        'win_streaks': np.array(win_streaks),
        'loss_streaks': np.array(loss_streaks),
        # NUOVO: Benchmark fields
        'benchmark_return': benchmark_return,
        'benchmark_drawdown': benchmark_drawdown,
        'below_benchmark_return_count': below_benchmark_return_count,
        'above_benchmark_drawdown_count': above_benchmark_drawdown_count,
        'prob_beat_benchmark_return': 1.0 - (below_benchmark_return_count / n_simulations) if benchmark_return is not None else None,
        'prob_beat_benchmark_dd': 1.0 - (above_benchmark_drawdown_count / n_simulations) if benchmark_drawdown is not None else None,        
        'stats': {
            'mean_return': np.mean(final_returns),
            'median_return': np.median(final_returns),
            'std_return': np.std(final_returns),
            'percentile_5': np.percentile(final_returns, 5),
            'percentile_25': np.percentile(final_returns, 25),
            'percentile_75': np.percentile(final_returns, 75),
            'percentile_95': np.percentile(final_returns, 95),
            'mean_dd': np.mean(max_drawdowns),
            'percentile_95_dd': np.percentile(max_drawdowns, 95),
            'mean_sharpe': np.mean(sharpe_ratios),
            'prob_positive': (np.array(final_returns) > 0).mean()
        }
    }


# =============================================================================
# 2. BLOCK BOOTSTRAP RETURNS
# =============================================================================

def monte_carlo_block_bootstrap(
    portfolio_returns: Union[pd.Series, np.ndarray],
    n_simulations: int = 10_000,
    block_size: int = 20,
    init_cash: float = 100_000,
    preserve_mean: bool = False,
    benchmark_return: Optional[float] = None,  # NUOVO
    benchmark_drawdown: Optional[float] = None,  # NUOVO
    random_seed: Optional[int] = None,
    show_progress: bool = True
) -> Dict:    
    
    """
    Monte Carlo Block Bootstrap su returns giornalieri.
    
    Resample blocchi contigui preservando autocorrelazione.
    UTILE per trend-following o momentum systems.
    
    Parametri
    ----------
    portfolio_returns : pd.Series or np.ndarray
        Returns giornalieri del portfolio
    n_simulations : int, default=10_000
        Numero simulazioni
    block_size : int, default=20
        Dimensione blocchi (~1 mese trading)
    init_cash : float, default=100_000
        Capitale iniziale
    preserve_mean : bool, default=False
        Se True, centra blocchi per preservare media
    random_seed : int, optional
        Seed riproducibilità
    show_progress : bool, default=True
        Progress bar
        
    Returns
    -------
    dict
        Risultati simulazioni
    """
    
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Converti a numpy
    if isinstance(portfolio_returns, pd.Series):
        returns = portfolio_returns.dropna().values
    else:
        returns = portfolio_returns[~np.isnan(portfolio_returns)]
    
    n_days = len(returns)
    n_blocks_needed = int(np.ceil(n_days / block_size))
    
    # Storage
    equity_curves = []
    final_returns = []
    max_drawdowns = []
    sharpe_ratios = []
        
    # NUOVO: Benchmark comparisons
    below_benchmark_return_count = 0
    above_benchmark_drawdown_count = 0

    pbar = tqdm(total=n_simulations, desc="Block Bootstrap", disable=not show_progress)
    
    for _ in range(n_simulations):
        # Bootstrap blocchi
        bootstrapped = []
        
        for _ in range(n_blocks_needed):
            start = np.random.randint(0, max(1, n_days - block_size))
            block = returns[start:start + block_size].copy()
            
            # Center se richiesto
            if preserve_mean:
                block = block - block.mean() + returns.mean()
            
            bootstrapped.extend(block)
        
        # Tronca
        bootstrapped = np.array(bootstrapped[:n_days])
        
        # Equity curve
        equity = init_cash * np.cumprod(1 + bootstrapped)
        equity = np.insert(equity, 0, init_cash)
        
        # Metriche
        final_return = (equity[-1] / init_cash) - 1
        
        cummax = np.maximum.accumulate(equity)
        drawdown = (equity - cummax) / cummax
        max_dd = abs(drawdown.min())
        
        sharpe = (np.mean(bootstrapped) / (np.std(bootstrapped) + 1e-8)) * np.sqrt(252)
        
        # Store
        equity_curves.append(equity)
        final_returns.append(final_return)
        max_drawdowns.append(max_dd)
        sharpe_ratios.append(sharpe)

        # NUOVO: Benchmark comparisons
        if benchmark_return is not None and final_return < benchmark_return:
            below_benchmark_return_count += 1
        if benchmark_drawdown is not None and max_dd > benchmark_drawdown:
            above_benchmark_drawdown_count += 1
        
        pbar.update(1)
    
    pbar.close()
    
    return {
        'method': 'Block Bootstrap',
        'n_simulations': n_simulations,
        'block_size': block_size,
        'n_days': n_days,
        'equity_curves': equity_curves,
        'final_returns': np.array(final_returns),
        'max_drawdowns': np.array(max_drawdowns),
        'sharpe_ratios': np.array(sharpe_ratios),

        # NUOVO: Benchmark fields
        'benchmark_return': benchmark_return,
        'benchmark_drawdown': benchmark_drawdown,
        'below_benchmark_return_count': below_benchmark_return_count,
        'above_benchmark_drawdown_count': above_benchmark_drawdown_count,
        'prob_beat_benchmark_return': 1.0 - (below_benchmark_return_count / n_simulations) if benchmark_return is not None else None,
        'prob_beat_benchmark_dd': 1.0 - (above_benchmark_drawdown_count / n_simulations) if benchmark_drawdown is not None else None,

        'stats': {
            'mean_return': np.mean(final_returns),
            'median_return': np.median(final_returns),
            'std_return': np.std(final_returns),
            'percentile_5': np.percentile(final_returns, 5),
            'percentile_95': np.percentile(final_returns, 95),
            'mean_dd': np.mean(max_drawdowns),
            'percentile_95_dd': np.percentile(max_drawdowns, 95),
            'mean_sharpe': np.mean(sharpe_ratios),
            'prob_positive': (np.array(final_returns) > 0).mean()
        }
    }


# =============================================================================
# 3. REGIME-SWITCHING BOOTSTRAP
# =============================================================================

def monte_carlo_regime_switching(
    portfolio_returns: Union[pd.Series, np.ndarray],
    n_simulations: int = 10_000,
    regime_window: int = 60,
    init_cash: float = 100_000,
    benchmark_return: Optional[float] = None,  # NUOVO
    benchmark_drawdown: Optional[float] = None,  # NUOVO
    random_seed: Optional[int] = None,
    show_progress: bool = True
) -> Dict:
    """
    Monte Carlo con identificazione regimi bull/bear.
    
    Resample returns preservando struttura regime.
    UTILE per strategies sensibili a bull/bear markets.
    
    Parametri
    ----------
    portfolio_returns : pd.Series or np.ndarray
        Returns giornalieri
    n_simulations : int, default=10_000
        Numero simulazioni
    regime_window : int, default=60
        Finestra rolling per identificare regime
    init_cash : float, default=100_000
        Capitale iniziale
    random_seed : int, optional
        Seed
    show_progress : bool, default=True
        Progress bar
        
    Returns
    -------
    dict
        Risultati
    """
    
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Converti
    if isinstance(portfolio_returns, pd.Series):
        returns = portfolio_returns.dropna().values
    else:
        returns = portfolio_returns[~np.isnan(portfolio_returns)]
    
    n_days = len(returns)
    
    # Identifica regimi (bull = 1, bear = 0)
    rolling_mean = pd.Series(returns).rolling(regime_window, min_periods=1).mean().values
    regime = (rolling_mean > 0).astype(int)
    
    # Separa returns per regime
    bull_returns = returns[regime == 1]
    bear_returns = returns[regime == 0]
    
    # Storage
    equity_curves = []
    final_returns = []
    max_drawdowns = []
    sharpe_ratios = []

    # NUOVO: Benchmark comparisons
    below_benchmark_return_count = 0
    above_benchmark_drawdown_count = 0
    
    pbar = tqdm(total=n_simulations, desc="Regime Switching", disable=not show_progress)
    
    for _ in range(n_simulations):
        sampled = []
        
        for r in regime:
            if r == 1 and len(bull_returns) > 0:
                sampled.append(np.random.choice(bull_returns))
            elif len(bear_returns) > 0:
                sampled.append(np.random.choice(bear_returns))
            else:
                sampled.append(0.0)
        
        sampled = np.array(sampled)
        
        # Equity
        equity = init_cash * np.cumprod(1 + sampled)
        equity = np.insert(equity, 0, init_cash)
        
        # Metriche
        final_return = (equity[-1] / init_cash) - 1
        
        cummax = np.maximum.accumulate(equity)
        drawdown = (equity - cummax) / cummax
        max_dd = abs(drawdown.min())
        
        sharpe = (np.mean(sampled) / (np.std(sampled) + 1e-8)) * np.sqrt(252)
        
        # Store
        equity_curves.append(equity)
        final_returns.append(final_return)
        max_drawdowns.append(max_dd)
        sharpe_ratios.append(sharpe)

        # NUOVO: Benchmark comparisons
        if benchmark_return is not None and final_return < benchmark_return:
            below_benchmark_return_count += 1
        if benchmark_drawdown is not None and max_dd > benchmark_drawdown:
            above_benchmark_drawdown_count += 1
        
        pbar.update(1)
    
    pbar.close()
    
    return {
        'method': 'Regime-Switching Bootstrap',
        'n_simulations': n_simulations,
        'regime_window': regime_window,
        'n_days': n_days,
        'pct_bull_days': (regime == 1).mean(),
        'equity_curves': equity_curves,
        'final_returns': np.array(final_returns),
        'max_drawdowns': np.array(max_drawdowns),
        'sharpe_ratios': np.array(sharpe_ratios),
        
        # NUOVO: Benchmark fields
        'benchmark_return': benchmark_return,
        'benchmark_drawdown': benchmark_drawdown,
        'below_benchmark_return_count': below_benchmark_return_count,
        'above_benchmark_drawdown_count': above_benchmark_drawdown_count,
        'prob_beat_benchmark_return': 1.0 - (below_benchmark_return_count / n_simulations) if benchmark_return is not None else None,
        'prob_beat_benchmark_dd': 1.0 - (above_benchmark_drawdown_count / n_simulations) if benchmark_drawdown is not None else None,

        'stats': {
            'mean_return': np.mean(final_returns),
            'median_return': np.median(final_returns),
            'std_return': np.std(final_returns),
            'percentile_5': np.percentile(final_returns, 5),
            'percentile_95': np.percentile(final_returns, 95),
            'mean_dd': np.mean(max_drawdowns),
            'percentile_95_dd': np.percentile(max_drawdowns, 95),
            'mean_sharpe': np.mean(sharpe_ratios),
            'prob_positive': (np.array(final_returns) > 0).mean()
        }
    }


# =============================================================================
# 4. SIMPLE BOOTSTRAP CORRECTED (BASELINE)
# =============================================================================

def monte_carlo_simple_corrected(
    portfolio_returns: Union[pd.Series, np.ndarray],
    n_simulations: int = 10_000,
    init_cash: float = 100_000,
    slippage: float = 0.0,
    shock_frequency: float = 0.0,
    shock_magnitude: Tuple[float, float] = (0.05, 0.15),
    benchmark_return: Optional[float] = None,  # NUOVO
    benchmark_drawdown: Optional[float] = None,  # NUOVO
    random_seed: Optional[int] = None,
    show_progress: bool = True
) -> Dict:    
    
    """
    Simple Bootstrap CORRETTO (fix del tuo originale).
    
    Baseline comparison con altri metodi.
    Include slippage e shock events opzionali.
    
    Parametri
    ----------
    portfolio_returns : pd.Series or np.ndarray
        Returns giornalieri
    n_simulations : int, default=10_000
        Numero simulazioni
    init_cash : float, default=100_000
        Capitale iniziale
    slippage : float, default=0.0
        Penalizzazione per trade (es. 0.0001 = 1bp)
    shock_frequency : float, default=0.0
        Percentuale giorni con shock (0-1)
    shock_magnitude : tuple, default=(0.05, 0.15)
        Range shock negativi
    random_seed : int, optional
        Seed
    show_progress : bool, default=True
        Progress bar
        
    Returns
    -------
    dict
        Risultati
    """
    
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Converti
    if isinstance(portfolio_returns, pd.Series):
        returns = portfolio_returns.dropna().values
    else:
        returns = portfolio_returns[~np.isnan(portfolio_returns)]
    
    n_days = len(returns)
    
    # Storage
    equity_curves = []
    final_returns = []
    max_drawdowns = []
    sharpe_ratios = []

    # NUOVO: Benchmark comparisons
    below_benchmark_return_count = 0
    above_benchmark_drawdown_count = 0
    
    pbar = tqdm(total=n_simulations, desc="Simple Bootstrap", disable=not show_progress)
    
    for _ in range(n_simulations):
        # Sample returns
        sampled = np.random.choice(returns, size=n_days, replace=True)
        
        # Slippage
        sampled -= slippage
        
        # Shock events (clustering)
        if shock_frequency > 0:
            n_shocks = int(n_days * shock_frequency)
            if n_shocks > 0:
                # Cluster shock
                n_clusters = max(1, n_shocks // 3)
                cluster_centers = np.random.choice(n_days, size=n_clusters, replace=False)
                
                shock_days = []
                for center in cluster_centers:
                    cluster_days = np.clip(
                        center + np.random.randint(-5, 6, size=3),
                        0, n_days - 1
                    )
                    shock_days.extend(cluster_days)
                
                shock_days = np.unique(shock_days)[:n_shocks]
                
                # Pareto-distributed shocks (fat tails)
                from scipy.stats import pareto
                shocks = pareto.rvs(1.5, size=n_shocks) * shock_magnitude[0]
                shocks = np.clip(shocks, shock_magnitude[0], shock_magnitude[1])
                
                sampled[shock_days] -= shocks
        
        # Equity
        equity = init_cash * np.cumprod(1 + sampled)
        equity = np.insert(equity, 0, init_cash)
        
        # Metriche (CORRETTE)
        final_return = (equity[-1] / init_cash) - 1
        
        cummax = np.maximum.accumulate(equity)
        drawdown = (equity - cummax) / cummax
        max_dd = abs(drawdown.min())
        
        # Sharpe CORRETTO (su sampled returns, non equity)
        sharpe = (np.mean(sampled) / (np.std(sampled) + 1e-8)) * np.sqrt(252)
        
        # Store
        equity_curves.append(equity)
        final_returns.append(final_return)
        max_drawdowns.append(max_dd)
        sharpe_ratios.append(sharpe)

        # NUOVO: Benchmark comparisons
        if benchmark_return is not None and final_return < benchmark_return:
            below_benchmark_return_count += 1
        if benchmark_drawdown is not None and max_dd > benchmark_drawdown:
            above_benchmark_drawdown_count += 1
        
        pbar.update(1)
    
    pbar.close()
    
    return {
        'method': 'Simple Bootstrap (Corrected)',
        'n_simulations': n_simulations,
        'n_days': n_days,
        'slippage': slippage,
        'shock_frequency': shock_frequency,
        'equity_curves': equity_curves,
        'final_returns': np.array(final_returns),
        'max_drawdowns': np.array(max_drawdowns),
        'sharpe_ratios': np.array(sharpe_ratios),
        # NUOVO: Benchmark fields
        'benchmark_return': benchmark_return,
        'benchmark_drawdown': benchmark_drawdown,
        'below_benchmark_return_count': below_benchmark_return_count,
        'above_benchmark_drawdown_count': above_benchmark_drawdown_count,
        'prob_beat_benchmark_return': 1.0 - (below_benchmark_return_count / n_simulations) if benchmark_return is not None else None,
        'prob_beat_benchmark_dd': 1.0 - (above_benchmark_drawdown_count / n_simulations) if benchmark_drawdown is not None else None,
        
        'stats': {
            'mean_return': np.mean(final_returns),
            'median_return': np.median(final_returns),
            'std_return': np.std(final_returns),
            'percentile_5': np.percentile(final_returns, 5),
            'percentile_95': np.percentile(final_returns, 95),
            'mean_dd': np.mean(max_drawdowns),
            'percentile_95_dd': np.percentile(max_drawdowns, 95),
            'mean_sharpe': np.mean(sharpe_ratios),
            'prob_positive': (np.array(final_returns) > 0).mean()
        }
    }


# =============================================================================
# ANALISI COMPARATIVA
# =============================================================================

def compare_mc_methods(
    mc_results_list: List[Dict],
    actual_portfolio_return: float,
    actual_portfolio_dd: float,
    actual_portfolio_sharpe: float,
    print_report: bool = True
) -> pd.DataFrame:
    """
    Confronta risultati di diversi metodi MC.
    
    Parametri
    ----------
    mc_results_list : list of dict
        Lista risultati da diverse funzioni MC
    actual_portfolio_return : float
        Return effettivo del portfolio (per confronto)
    actual_portfolio_dd : float
        Drawdown effettivo (valore assoluto)
    actual_portfolio_sharpe : float
        Sharpe effettivo
    print_report : bool, default=True
        Stampa report comparativo
        
    Returns
    -------
    pd.DataFrame
        Tabella comparativa metriche
    """
    
    comparison = []
    
    for mc_result in mc_results_list:
        method = mc_result['method']
        stats = mc_result['stats']
        
        # Percentile actual return
        returns_dist = mc_result['final_returns']
        percentile_return = (returns_dist < actual_portfolio_return).mean()
        
        # Percentile actual DD
        dd_dist = mc_result['max_drawdowns']
        percentile_dd = (dd_dist < actual_portfolio_dd).mean()
        
        # Percentile actual Sharpe
        sharpe_dist = mc_result['sharpe_ratios']
        percentile_sharpe = (sharpe_dist < actual_portfolio_sharpe).mean()
        
        comparison.append({
            'Method': method,
            'Mean Return': stats['mean_return'],
            'Median Return': stats['median_return'],
            'Std Return': stats['std_return'],
            'P5 Return': stats['percentile_5'],
            'P95 Return': stats['percentile_95'],
            'Mean DD': stats['mean_dd'],
            'P95 DD': stats['percentile_95_dd'],
            'Mean Sharpe': stats['mean_sharpe'],
            'Prob Positive': stats['prob_positive'],
            'Actual Return Percentile': percentile_return,
            'Actual DD Percentile': percentile_dd,
            'Actual Sharpe Percentile': percentile_sharpe
        })
    
    df = pd.DataFrame(comparison)
    
    if print_report:
        print("=" * 100)
        print("MONTE CARLO METHODS COMPARISON")
        print("=" * 100)
        print(f"\nActual Portfolio Performance:")
        print(f"  Return: {actual_portfolio_return:.2%}")
        print(f"  Max DD: {actual_portfolio_dd:.2%}")
        print(f"  Sharpe: {actual_portfolio_sharpe:.3f}")
        print()
        
        # Tabella
        # print(df.to_string(index=False))
        my_display(df)
        print()
        
        print("=" * 100)
        print("INTERPRETATION")
        print("=" * 100)
        
        # Analizza discrepanze
        percentiles_return = df['Actual Return Percentile'].values
        if percentiles_return.max() - percentiles_return.min() > 0.20:
            print("⚠️  HIGH VARIANCE between methods (>20pp difference)")
            print("   → Suggests autocorrelation or regime-dependency in returns")
            print("   → Prefer Block Bootstrap or Regime-Switching results")
        else:
            print("✅ CONSISTENT results across methods")
        
        print()
        
        # Check overfitting
        avg_percentile = percentiles_return.mean()
        if avg_percentile > 0.85:
            print("🚩 Actual return > 85th percentile MC")
            print("   → Possible overfitting or luck")
            print("   → Expect mean reversion in live trading")
        elif avg_percentile < 0.25:
            print("⚠️  Actual return < 25th percentile MC")
            print("   → Below average performance vs MC")
        else:
            print("✅ Actual performance within normal MC range")
        
        print("=" * 100)
    
    return df


# =============================================================================
# VISUALIZZAZIONI
# =============================================================================

def plot_mc_results(
    mc_result: Dict,
    actual_portfolio_return: float,
    actual_portfolio_dd: float,
    benchmark_return: Optional[float] = None,  # NUOVO
    benchmark_dd: Optional[float] = None,  # NUOVO
    save_path: Optional[str] = None,
    figsize: Tuple[int, int] = (16, 12)
):    
    
    """
    Visualizza risultati singolo metodo MC.
    
    4 grafici:
    1. Equity curves sample
    2. Return distribution
    3. Drawdown distribution
    4. Sharpe distribution
    """
    
    method = mc_result['method']
    equity_curves = mc_result['equity_curves']
    final_returns = mc_result['final_returns']
    max_drawdowns = mc_result['max_drawdowns']
    sharpe_ratios = mc_result['sharpe_ratios']
    
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    
    # =========================================================================
    # PLOT 1: Equity Curves Sample
    # =========================================================================
    ax = axes[0, 0]
    
    # Plot sample curves
    n_plot = min(50, len(equity_curves))
    for i in range(n_plot):
        ax.plot(equity_curves[i], alpha=0.2, linewidth=0.8, color='steelblue')
    
    # Median curve
    equity_array = np.array([eq for eq in equity_curves])
    median_curve = np.median(equity_array, axis=0)
    ax.plot(median_curve, 'r-', linewidth=2.5, label='MC Median', zorder=10)
    
    ax.set_title(f'{method} - Equity Curves (sample)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Days')
    ax.set_ylabel('Portfolio Value ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 2: Return Distribution
    # =========================================================================
    ax = axes[0, 1]
    
    ax.hist(final_returns * 100, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(np.percentile(final_returns, 5) * 100, color='red', linestyle='--', 
               linewidth=2, label=f'5th pct: {np.percentile(final_returns, 5):.1%}')
    ax.axvline(np.mean(final_returns) * 100, color='blue', linestyle='-',
               linewidth=2, label=f'Mean: {np.mean(final_returns):.1%}')
    ax.axvline(np.percentile(final_returns, 95) * 100, color='green', linestyle='--',
               linewidth=2, label=f'95th pct: {np.percentile(final_returns, 95):.1%}')
    ax.axvline(actual_portfolio_return * 100, color='purple', linestyle='-',
               linewidth=2.5, label=f'Actual: {actual_portfolio_return:.1%}')
    
    # NUOVO: Benchmark line
    if benchmark_return is not None:
        ax.axvline(benchmark_return * 100, color='purple', linestyle='-',
                   linewidth=2.5, label=f'Benchmark: {benchmark_return:.1%}')
        
        # Shade "beat benchmark" zone
        beat_bench_returns = final_returns[final_returns > benchmark_return]
        if len(beat_bench_returns) > 0:
            ax.hist(beat_bench_returns * 100, bins=30, alpha=0.3, color='green',
                   label=f'Beat Bench: {len(beat_bench_returns)/len(final_returns):.1%}')

    ax.set_title('Final Return Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Return (%)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 3: Drawdown Distribution
    # =========================================================================
    ax = axes[1, 0]
    
    ax.hist(max_drawdowns * 100, bins=50, alpha=0.7, color='salmon', edgecolor='black')
    ax.axvline(np.percentile(max_drawdowns, 5) * 100, color='green', linestyle='--',
               linewidth=2, label=f'5th pct: {np.percentile(max_drawdowns, 5):.1%}')
    ax.axvline(np.mean(max_drawdowns) * 100, color='blue', linestyle='-',
               linewidth=2, label=f'Mean: {np.mean(max_drawdowns):.1%}')
    ax.axvline(np.percentile(max_drawdowns, 95) * 100, color='red', linestyle='--',
               linewidth=2, label=f'95th pct: {np.percentile(max_drawdowns, 95):.1%}')
    ax.axvline(actual_portfolio_dd * 100, color='purple', linestyle='-',
               linewidth=2.5, label=f'Actual: {actual_portfolio_dd:.1%}')
    # NUOVO: Benchmark line
    if benchmark_return is not None:
        ax.axvline(benchmark_return * 100, color='purple', linestyle='-',
                   linewidth=2.5, label=f'Benchmark: {benchmark_return:.1%}')
        
        # Shade "beat benchmark" zone
        beat_bench_returns = final_returns[final_returns > benchmark_return]
        if len(beat_bench_returns) > 0:
            ax.hist(beat_bench_returns * 100, bins=30, alpha=0.3, color='green',
                   label=f'Beat Bench: {len(beat_bench_returns)/len(final_returns):.1%}')
    
    ax.set_title('Max Drawdown Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Max Drawdown (%)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 4: Sharpe Distribution
    # =========================================================================
    ax = axes[1, 1]
    
    ax.hist(sharpe_ratios, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
    ax.axvline(np.percentile(sharpe_ratios, 5), color='red', linestyle='--',
               linewidth=2, label=f'5th pct: {np.percentile(sharpe_ratios, 5):.2f}')
    ax.axvline(np.mean(sharpe_ratios), color='blue', linestyle='-',
               linewidth=2, label=f'Mean: {np.mean(sharpe_ratios):.2f}')
    ax.axvline(np.percentile(sharpe_ratios, 95), color='green', linestyle='--',
               linewidth=2, label=f'95th pct: {np.percentile(sharpe_ratios, 95):.2f}')
    
    ax.set_title('Sharpe Ratio Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Sharpe Ratio')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved: {save_path}")
    
    plt.show()
    
    return fig


def plot_mc_comparison(
    mc_results_list: List[Dict],
    actual_portfolio_return: float,
    actual_portfolio_dd: float,
    save_path: Optional[str] = None,
    figsize: Tuple[int, int] = (16, 10)
):
    """
    Confronto visivo tra metodi MC diversi.
    
    3 grafici comparativi side-by-side.
    """
    
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    colors = ['steelblue', 'coral', 'lightgreen', 'plum']
    
    # =========================================================================
    # PLOT 1: Return Distributions
    # =========================================================================
    ax = axes[0]
    
    for i, mc_result in enumerate(mc_results_list):
        method = mc_result['method']
        returns = mc_result['final_returns'] * 100
        
        ax.hist(returns, bins=30, alpha=0.5, color=colors[i % len(colors)],
                label=method, edgecolor='black')
    
    ax.axvline(actual_portfolio_return * 100, color='red', linestyle='--',
               linewidth=2.5, label='Actual')
    
    ax.set_title('Return Distributions Comparison', fontsize=14, fontweight='bold')
    ax.set_xlabel('Final Return (%)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 2: DD Distributions
    # =========================================================================
    ax = axes[1]
    
    for i, mc_result in enumerate(mc_results_list):
        method = mc_result['method']
        dds = mc_result['max_drawdowns'] * 100
        
        ax.hist(dds, bins=30, alpha=0.5, color=colors[i % len(colors)],
                label=method, edgecolor='black')
    
    ax.axvline(actual_portfolio_dd * 100, color='red', linestyle='--',
               linewidth=2.5, label='Actual')
    
    ax.set_title('Drawdown Distributions Comparison', fontsize=14, fontweight='bold')
    ax.set_xlabel('Max Drawdown (%)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 3: Percentile Comparison
    # =========================================================================
    ax = axes[2]
    
    methods = [mc['method'] for mc in mc_results_list]
    x_pos = np.arange(len(methods))
    
    # Return percentiles
    return_percentiles = []
    dd_percentiles = []
    
    for mc_result in mc_results_list:
        ret_pct = (mc_result['final_returns'] < actual_portfolio_return).mean()
        dd_pct = (mc_result['max_drawdowns'] < actual_portfolio_dd).mean()
        
        return_percentiles.append(ret_pct * 100)
        dd_percentiles.append(dd_pct * 100)
    
    width = 0.35
    ax.bar(x_pos - width/2, return_percentiles, width, label='Return Percentile',
           color='steelblue', edgecolor='black')
    ax.bar(x_pos + width/2, dd_percentiles, width, label='DD Percentile',
           color='coral', edgecolor='black')
    
    ax.axhline(85, color='red', linestyle='--', alpha=0.5, label='85th (overfitting risk)')
    ax.axhline(50, color='gray', linestyle='-', alpha=0.5)
    ax.axhline(25, color='orange', linestyle='--', alpha=0.5, label='25th (below avg)')
    
    ax.set_title('Actual Performance Percentile by Method', fontsize=14, fontweight='bold')
    ax.set_ylabel('Percentile (%)')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(methods, rotation=15, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Comparison plot saved: {save_path}")
    
    plt.show()
    
    return fig


# =============================================================================
# HELPERS
# =============================================================================

def _max_consecutive(mask):
    """Calcola massima sequenza consecutiva di True."""
    if len(mask) == 0:
        return 0
    
    max_streak = 0
    current_streak = 0
    
    for val in mask:
        if val:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0
    
    return max_streak


# =============================================================================
# ESEMPIO USO
# =============================================================================

if __name__ == "__main__":
    print("Monte Carlo Trading Systems - Example Usage")
    print("=" * 60)
    print()
    print("ESEMPIO PIPELINE:")
    print("-" * 60)
    print("""
# Dopo aver creato portfolio VectorBT

# 1. METODO RACCOMANDATO: Trade-Based Bootstrap
mc_trade = monte_carlo_trade_bootstrap(
    trades_df=portfolio.trades.records_readable,
    n_simulations=10_000,
    init_cash=100_000,
    random_seed=42
)

# 2. Block Bootstrap (per trend-following)
mc_block = monte_carlo_block_bootstrap(
    portfolio_returns=portfolio.returns(),
    n_simulations=10_000,
    block_size=20,
    random_seed=42
)

# 3. Regime-Switching (se sensibile a bull/bear)
mc_regime = monte_carlo_regime_switching(
    portfolio_returns=portfolio.returns(),
    n_simulations=10_000,
    random_seed=42
)

# 4. Simple Bootstrap (baseline)
mc_simple = monte_carlo_simple_corrected(
    portfolio_returns=portfolio.returns(),
    n_simulations=10_000,
    slippage=0.0001,
    shock_frequency=0.002,
    random_seed=42
)

# 5. Confronta tutti i metodi
actual_return = portfolio.total_return()
actual_dd = abs(portfolio.max_drawdown())
actual_sharpe = portfolio.sharpe_ratio()

comparison = compare_mc_methods(
    mc_results_list=[mc_trade, mc_block, mc_regime, mc_simple],
    actual_portfolio_return=actual_return,
    actual_portfolio_dd=actual_dd,
    actual_portfolio_sharpe=actual_sharpe,
    print_report=True
)

# 6. Visualizza singolo metodo
plot_mc_results(mc_trade, actual_return, actual_dd, save_path='./mc_trade.png')

# 7. Confronto visivo
plot_mc_comparison(
    [mc_trade, mc_block, mc_regime, mc_simple],
    actual_return,
    actual_dd,
    save_path='./mc_comparison.png'
)
""")
    print("-" * 60)

Monte Carlo Trading Systems - Example Usage

ESEMPIO PIPELINE:
------------------------------------------------------------

# Dopo aver creato portfolio VectorBT

# 1. METODO RACCOMANDATO: Trade-Based Bootstrap
mc_trade = monte_carlo_trade_bootstrap(
    trades_df=portfolio.trades.records_readable,
    n_simulations=10_000,
    init_cash=100_000,
    random_seed=42
)

# 2. Block Bootstrap (per trend-following)
mc_block = monte_carlo_block_bootstrap(
    portfolio_returns=portfolio.returns(),
    n_simulations=10_000,
    block_size=20,
    random_seed=42
)

# 3. Regime-Switching (se sensibile a bull/bear)
mc_regime = monte_carlo_regime_switching(
    portfolio_returns=portfolio.returns(),
    n_simulations=10_000,
    random_seed=42
)

# 4. Simple Bootstrap (baseline)
mc_simple = monte_carlo_simple_corrected(
    portfolio_returns=portfolio.returns(),
    n_simulations=10_000,
    slippage=0.0001,
    shock_frequency=0.002,
    random_seed=42
)

# 5. Confronta tutti i metodi
actual_re